In [4]:
!pip install -q langgraph langchain-groq langchain-community langchain-text-splitters chromadb sentence-transformers pypdf python-docx

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 710.3 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 54.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 55.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9

In [5]:
# LangGraph
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages

# Typing
from typing import Annotated
from typing_extensions import TypedDict

# LangChain
from langchain_groq import ChatGroq
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import SentenceTransformerEmbeddings
from langchain_core.messages import HumanMessage, SystemMessage

# File handling
from google.colab import files, userdata
from docx import Document
from pypdf import PdfReader
import io

print("✅ All imports done!")

/tmp/ipykernel_2183/1956840851.py:12: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


✅ All imports done!


setup LLM

In [6]:
llm = ChatGroq(
    api_key=userdata.get('GroqKey'),
    model="llama-3.3-70b-versatile"
)

print("✅ LLM ready!")

✅ LLM ready!


define states


In [7]:
class InterviewState(TypedDict):
    # Documents
    cv_text: str
    jd_text: str

    # Agent 1 output
    extracted_skills: str

    # RAG context
    retrieved_context: str

    # Interview progress
    current_question: str
    user_answer: str
    score: int
    feedback: str
    question_count: int
    total_score: int
    asked_questions: list

    # Final output
    final_report: str

    # Memory
    messages: Annotated[list, add_messages]

print("✅ State defined!")

✅ State defined!


upload cv

In [8]:
print("📄 Step 1: Upload your CV file (.docx, .pdf, or .txt)")
uploaded_cv = files.upload()

cv_filename = list(uploaded_cv.keys())[0]
cv_content = uploaded_cv[cv_filename]

# Read CV based on file type
if cv_filename.endswith(".txt"):
    cv_text = cv_content.decode("utf-8")
elif cv_filename.endswith(".pdf"):
    from pypdf import PdfReader
    reader = PdfReader(io.BytesIO(cv_content))
    cv_text = ""
    for page in reader.pages:
        cv_text += page.extract_text()
elif cv_filename.endswith(".docx"):
    doc = Document(io.BytesIO(cv_content))
    cv_text = ""
    for paragraph in doc.paragraphs:
        cv_text += paragraph.text + "\n"

print(f"✅ CV uploaded: {cv_filename}")
print(f"\n📝 CV Preview:\n{cv_text[:300]}...")

📄 Step 1: Upload your CV file (.docx, .pdf, or .txt)


Saving Sonaina_Ilyas_Resume_Simera.docx to Sonaina_Ilyas_Resume_Simera.docx
✅ CV uploaded: Sonaina_Ilyas_Resume_Simera.docx

📝 CV Preview:
Sonaina Ilyas
Islamabad, Pakistan (Open to Remote)  |  sonainailyas005@gmail.com  |  +92 324 9752902
linkedin.com/in/sonaina-ilyas-3a0b38331
PROFESSIONAL SUMMARY

Organized, detail-focused administrative professional with an MS in Computer Science and three years of running the day-to-day administra...


upload job description

In [9]:
print("📋 Step 2: Upload Job Description file (.docx, .pdf, or .txt)")
uploaded_jd = files.upload()

jd_filename = list(uploaded_jd.keys())[0]
jd_content = uploaded_jd[jd_filename]

# Read job description based on file type
if jd_filename.endswith(".txt"):
    jd_text = jd_content.decode("utf-8")
elif jd_filename.endswith(".pdf"):
    from pypdf import PdfReader
    reader = PdfReader(io.BytesIO(jd_content))
    jd_text = ""
    for page in reader.pages:
        jd_text += page.extract_text()
elif jd_filename.endswith(".docx"):
    doc = Document(io.BytesIO(jd_content))
    jd_text = ""
    for paragraph in doc.paragraphs:
        jd_text += paragraph.text + "\n"

print(f"✅ Job Description uploaded: {jd_filename}")
print(f"\n📝 Job Description Preview:\n{jd_text[:300]}...")

📋 Step 2: Upload Job Description file (.docx, .pdf, or .txt)


Saving jobDes.txt to jobDes.txt
✅ Job Description uploaded: jobDes.txt

📝 Job Description Preview:
Job Title: Data Analyst
Company: ABC Company

Requirements:
- 2+ years experience in data analysis
- Python and SQL skills
- Experience with Tableau or Power BI
- Strong communication skills
- AI and Machine Learning knowledge preferred

Responsibilities:
- Analyze large datasets
- Creat...


RAG

In [10]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

cv_chunks = text_splitter.create_documents(
    texts=[cv_text],
    metadatas=[{"source": "cv"}]
)

jd_chunks = text_splitter.create_documents(
    texts=[jd_text],
    metadatas=[{"source": "job_description"}]
)

all_chunks = cv_chunks + jd_chunks

embeddings = SentenceTransformerEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

vectorstore = Chroma.from_documents(
    documents=all_chunks,
    embedding=embeddings
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print(f"✅ CV split into {len(cv_chunks)} chunks!")
print(f"✅ JD split into {len(jd_chunks)} chunks!")
print("✅ RAG ready!")

/tmp/ipykernel_2183/2178474678.py:18: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = SentenceTransformerEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ CV split into 14 chunks!
✅ JD split into 1 chunks!
✅ RAG ready!


node 1 Resume Analyzer

In [11]:
def analyze_resume(state: InterviewState):
    print("📄 Node 1: Analyzing resume...")

    prompt = f"""
    Analyze this CV and Job Description:

    CV: {state['cv_text']}

    Job Description: {state['jd_text']}

    Please extract:
    1. Key skills from CV
    2. Job requirements from JD
    3. Matching skills
    4. Missing skills/gaps
    """

    response = llm.invoke(prompt)

    return {"extracted_skills": response.content}

print("✅ Node 1 ready!")

✅ Node 1 ready!


RAG Retriever

In [12]:
def retrieve_knowledge(state: InterviewState):
    print("🔍 Node 2: Retrieving knowledge...")

    # Build search query from extracted skills
    query = f"Interview questions for: {state['extracted_skills']}"

    # Search RAG
    docs = retriever.invoke(query)

    # Combine chunks into one text
    context = "\n\n".join([doc.page_content for doc in docs])

    print(f"📄 Retrieved {len(docs)} relevant chunks!")

    return {"retrieved_context": context}

print("✅ Node 2 ready!")

✅ Node 2 ready!


In [13]:
def interview_loop(state: InterviewState):
    print("🎯 Node 3: Interview Loop...")

    # Part 1 - Generate question
    question_prompt = f"""
    Based on this context from CV and Job Description:
    {state['retrieved_context']}

    Candidate skills: {state['extracted_skills']}
    Questions already asked: {state['asked_questions']}

    Generate ONE technical interview question.
    Do not repeat previous questions.
    Make it relevant to the job requirements.
    Only return the question, nothing else.
    """
    question = llm.invoke(question_prompt).content
    print(f"\n🤔 Question {state['question_count'] + 1}: {question}")

    # Part 2 - Get user answer
    user_answer = input("\nYour answer: ")

    # Part 3 - Evaluate answer
    eval_prompt = f"""
    Question: {question}
    Candidate answer: {user_answer}
    Job requirements: {state['jd_text']}

    Evaluate the answer and reply in EXACTLY this format:
    SCORE: [number 1-10]
    FEEDBACK: [your detailed feedback]
    """
    evaluation = llm.invoke(eval_prompt).content

    # Part 4 - Extract score and feedback
    score = 0
    feedback = ""
    for line in evaluation.split("\n"):
        if "SCORE:" in line:
            score = int(''.join(filter(str.isdigit, line.split("SCORE:")[1])))
        if "FEEDBACK:" in line:
            feedback = line.split("FEEDBACK:")[1].strip()

    # Part 5 - Good answer or give hint?
    if score >= 6:
        print(f"\n✅ Good answer! Score: {score}/10")
        print(f"💬 Feedback: {feedback}")
    else:
        print(f"\n💡 Score: {score}/10 — Let me give you a hint:")
        hint_prompt = f"""
        The candidate struggled with this question:
        {question}

        Their answer was: {user_answer}

        Give a helpful hint without giving away the full answer.
        """
        hint = llm.invoke(hint_prompt).content
        print(f"💡 Hint: {hint}")

    # Part 6 - Update state
    return {
        "current_question": question,
        "user_answer": user_answer,
        "score": score,
        "feedback": feedback,
        "question_count": state['question_count'] + 1,
        "total_score": state['total_score'] + score,
        "asked_questions": state['asked_questions'] + [question],
        "messages": [HumanMessage(content=user_answer)]
    }

print("✅ Node 3 ready!")

✅ Node 3 ready!


Performance Report Generator

In [14]:
def generate_report(state: InterviewState):
    print("📊 Node 4: Generating final report...")

    report_prompt = f"""
    Generate a professional interview performance report:

    Candidate Skills: {state['extracted_skills']}
    Total Score: {state['total_score']} out of {state['question_count'] * 10}
    Questions Asked: {state['asked_questions']}
    Last Feedback: {state['feedback']}
    Job Requirements: {state['jd_text']}

    Include in the report:
    1. 📊 Overall performance score
    2. ✅ Strong areas
    3. ❌ Weak areas
    4. 📚 Topics to study
    5. 🎯 Final recommendation
    """

    report = llm.invoke(report_prompt).content

    print("\n" + "="*50)
    print("📄 FINAL INTERVIEW REPORT:")
    print("="*50)
    print(report)

    return {"final_report": report}

print("✅ Node 4 ready!")

✅ Node 4 ready!


Conditional Function

In [15]:
def should_continue(state: InterviewState):
    if state['question_count'] >= 5:
        return "generate_report"  # go to report
    else:
        return "retrieve_knowledge"  # loop back

In [16]:
graph = StateGraph(InterviewState)

# Add all 4 nodes
graph.add_node("analyze_resume", analyze_resume)
graph.add_node("retrieve_knowledge", retrieve_knowledge)
graph.add_node("interview_loop", interview_loop)
graph.add_node("generate_report", generate_report)

# Entry point
graph.set_entry_point("analyze_resume")

# Normal edges
graph.add_edge("analyze_resume", "retrieve_knowledge")
graph.add_edge("retrieve_knowledge", "interview_loop")

# Conditional edge ← after interview loop
graph.add_conditional_edges(
    "interview_loop",
    should_continue,
    {
        "generate_report": "generate_report",
        "retrieve_knowledge": "retrieve_knowledge"
    }
)

# End
graph.add_edge("generate_report", END)

# Compile
app = graph.compile()

print("✅ Graph ready!")

✅ Graph ready!


running interview copilot

In [17]:
# Initial state
initial_state = {
    "cv_text": cv_text,
    "jd_text": jd_text,
    "extracted_skills": "",
    "retrieved_context": "",
    "current_question": "",
    "user_answer": "",
    "score": 0,
    "feedback": "",
    "question_count": 0,
    "total_score": 0,
    "asked_questions": [],
    "final_report": "",
    "messages": []
}

print("🚀 Starting AI Interview Copilot!")
print("="*50)
print("📄 Analyzing your CV and Job Description...")
print("🎯 Get ready for your interview!")
print("="*50)

# Run the graph
result = app.invoke(initial_state)

print("\n✅ Interview Complete!")

🚀 Starting AI Interview Copilot!
📄 Analyzing your CV and Job Description...
🎯 Get ready for your interview!
📄 Node 1: Analyzing resume...
🔍 Node 2: Retrieving knowledge...
📄 Retrieved 3 relevant chunks!
🎯 Node 3: Interview Loop...

🤔 Question 1: How would you use SQL to optimize the performance of a slow-running query that is retrieving a large dataset from a database, and what indexing strategies would you consider to improve query execution time?

Your answer: i dont know

💡 Score: 1/10 — Let me give you a hint:
💡 Hint: When optimizing a slow-running query, it's essential to understand that SQL provides various techniques to improve performance. Here's a hint to get you started: 

Consider the following steps: 
1. **Analyze the query**: Look at the query structure and identify potential bottlenecks, such as subqueries, joins, or sorting operations.
2. **Examine indexing options**: Think about the types of indexes that can be applied to the columns used in the query's WHERE, JOIN, and